<a href="https://colab.research.google.com/github/dewinafiah898-collab/komputasi-numerik/blob/main/numerik0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# SPL contoh:
#  2x +  y -  z =   8
# -3x -  y + 2z = -11
# -2x +  y + 2z =  -3

A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)   # matriks koefisien

b = np.array([8, -11, -3], dtype=float)     # vektor konstanta


# **PENJELASAN :**
numpy dipakai untuk operasi matriks. A menyimpan koefisien variabel dan b menyimpan ruas kanan persamaan. Gunakan dtype=float agar pembagian tidak terpotong menjadi bilangan bulat. Ubah isi A dan b untuk SPL lain (harus berukuran n×n).bold text

In [ ]:
def tampil(M, judul=""):
    if judul:
        print(judul)
    for baris in M:
        print("  ".join(f"{v:9.3f}" for v in baris))
    print()

**PENJELASAN:**
fungsi bantu untuk mencetak matriks dengan rapi (3 angka di belakang koma), sehingga langkah-langkah eliminasi mudah dibaca.

In [ ]:
def eliminasi_maju(A, b, tampilkan=True):
    n = len(b)
    M = np.hstack((A, b.reshape(-1, 1)))   # matriks augmented [A | b]

    if tampilkan:
        tampil(M, "Matriks augmented awal:")

    for k in range(n):
        # 1. Pivoting: cari baris dengan |nilai| terbesar pada kolom k
        p = k + np.argmax(np.abs(M[k:, k]))

        # Jika pivot ~ 0, lewati kolom ini
        if abs(M[p, k]) < 1e-12:
            continue

        # 2. Tukar baris k dengan baris pivot
        if p != k:
            M[[k, p]] = M[[p, k]]
            if tampilkan:
                tampil(M, f"Tukar baris {k+1} <-> {p+1}:")

        # 3. Nolkan semua elemen di bawah pivot
        for i in range(k + 1, n):
            faktor = M[i, k] / M[k, k]
            M[i, k:] = M[i, k:] - faktor * M[k, k:]

        if tampilkan:
            tampil(M, f"Setelah eliminasi kolom {k+1}:")

    return M

**PENJELASAN :**
ini inti metode Gauss. Matriks A digabung dengan b menjadi matriks augmented, lalu diubah menjadi bentuk segitiga atas.

Pivoting: memilih elemen terbesar pada kolom sebagai pivot agar lebih stabil secara numerik dan menghindari pembagian dengan nol.
Tukar baris: memindahkan baris pivot ke posisi k.
Eliminasi: untuk tiap baris di bawah pivot, hitung faktor = M[i,k] / M[k,k], lalu kurangkan faktor × baris pivot dari baris tersebut sehingga elemen di bawah pivot menjadi 0.

In [ ]:
def substitusi_mundur(M):
    n = M.shape[0]
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):          # dari baris terakhir ke atas
        jumlah = np.dot(M[i, i+1:n], x[i+1:n])
        x[i] = (M[i, n] - jumlah) / M[i, i]

    return x

**PENJELASAN :**
setelah matriks berbentuk segitiga atas, solusi dicari dari baris paling bawah. Baris terakhir memberi nilai x_n langsung. Nilai itu disubstitusikan ke baris di atasnya untuk mendapat x_(n-1), dan seterusnya sampai x_1.

In [ ]:
def gauss(A, b, tampilkan=True):
    n = len(b)
    M = eliminasi_maju(A, b, tampilkan)

    for i in range(n):
        koefisien_nol = np.all(np.abs(M[i, :n]) < 1e-12)
        konstanta_nol = abs(M[i, n]) < 1e-12

        if koefisien_nol and not konstanta_nol:
            print("Hasil: SPL TIDAK memiliki solusi.")
            return None
        if koefisien_nol and konstanta_nol:
            print("Hasil: SPL memiliki SOLUSI TAK HINGGA BANYAK.")
            return None

    x = substitusi_mundur(M)
    print("Hasil: SPL memiliki SOLUSI TUNGGAL.")
    return x

**PENJELASAN** :
fungsi ini menggabungkan semua langkah dan memeriksa jenis solusi.

Baris 0 0 0 | c dengan c ≠ 0 berarti tidak ada solusi.
Baris 0 0 0 | 0 berarti solusi tak hingga banyak.
Jika tidak ada baris bermasalah, SPL punya solusi tunggal dan dihitung dengan substitusi mundur.

In [ ]:
x = gauss(A, b)

if x is not None:
    print("Solusi:")
    for i, nilai in enumerate(x, start=1):
        print(f"x{i} = {nilai:.4f}")

**PENJELASAN** :
memanggil fungsi gauss pada SPL di Sel 1. Untuk contoh ini, hasilnya x1 = 2, x2 = 3, x3 = -1.

In [ ]:
# Cek 1: substitusi balik, A·x harus sama dengan b
print("A·x =", A @ x)
print("b   =", b)

# Cek 2: bandingkan dengan fungsi bawaan numpy
print("numpy:", np.linalg.solve(A, b))
print("Sama?", np.allclose(x, np.linalg.solve(A, b)))

**PENJELASAN :**
memastikan solusi benar. A @ x harus menghasilkan b, dan hasil Gauss harus sama dengan np.linalg.solve (True jika sama).

In [ ]:
# Tidak ada solusi
A1 = np.array([[1, 1], [2, 2]], dtype=float)
b1 = np.array([3, 7], dtype=float)
gauss(A1, b1, tampilkan=False)

# Solusi tak hingga
A2 = np.array([[1, 1], [2, 2]], dtype=float)
b2 = np.array([3, 6], dtype=float)
gauss(A2, b2, tampilkan=False)

**PENJELASAN :**
menguji dua kasus khusus. Pada A1, persamaan kedua adalah 2× ruas kiri persamaan pertama tetapi konstantanya tidak cocok (7 ≠ 6), sehingga tidak ada solusi. Pada A2, persamaan kedua identik dengan kelipatan persamaan pertama, sehingga solusinya tak hingga banyak.